# Multidimensional data frames: Using PySpark with JSON data.

DataFrames can contain arrays, maps, and structs

In [1]:
# reading json with python
import json

sample_json = """{
  "id": 143,
  "name": "Silicon Valley",
  "type": "Scripted",
  "language": "English",
  "genres": [
    "Comedy"
  ],
  "network": {
    "id": 8,
    "name": "HBO",
    "country": {
      "name": "United States",
      "code": "US",
      "timezone": "America/New_York"
    }
  }
}"""

document = json.loads(sample_json)
print(document)
type(document)

{'id': 143, 'name': 'Silicon Valley', 'type': 'Scripted', 'language': 'English', 'genres': ['Comedy'], 'network': {'id': 8, 'name': 'HBO', 'country': {'name': 'United States', 'code': 'US', 'timezone': 'America/New_York'}}}


dict

In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F 

spark = SparkSession.builder.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/01/05 04:28:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# read a show file from a jsonLine doc with a single show
shows = spark.read.json("./data/shows/shows-silicon-valley.json")
shows.show()

+--------------------+--------------------+--------------------+--------+---+--------------------+--------+--------------+--------------------+--------------------+----------+------+-------+-----------------+------+--------------------+--------+----------+--------------------+----------+------+
|           _embedded|              _links|           externals|  genres| id|               image|language|          name|             network|        officialSite| premiered|rating|runtime|         schedule|status|             summary|    type|   updated|                 url|webChannel|weight|
+--------------------+--------------------+--------------------+--------+---+--------------------+--------+--------------+--------------------+--------------------+----------+------+-------+-----------------+------+--------------------+--------+----------+--------------------+----------+------+
|{[{{{http://api.t...|{{http://api.tvma...|{tt2575988, 27716...|[Comedy]|143|{http://static.tv...| English|Silic

In [4]:
# read three shows
three_shows = spark.read.json("./data/shows/shows-*.json", multiLine=True)
print(three_shows.count())
# assert three_shows.count() == 3

1


In [5]:
shows.printSchema()

root
 |-- _embedded: struct (nullable = true)
 |    |-- episodes: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- _links: struct (nullable = true)
 |    |    |    |    |-- self: struct (nullable = true)
 |    |    |    |    |    |-- href: string (nullable = true)
 |    |    |    |-- airdate: string (nullable = true)
 |    |    |    |-- airstamp: string (nullable = true)
 |    |    |    |-- airtime: string (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- image: struct (nullable = true)
 |    |    |    |    |-- medium: string (nullable = true)
 |    |    |    |    |-- original: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- number: long (nullable = true)
 |    |    |    |-- runtime: long (nullable = true)
 |    |    |    |-- season: long (nullable = true)
 |    |    |    |-- summary: string (nullable = true)
 |    |    |    |-- url: string (nullable = true

In [6]:
print(shows.columns)

['_embedded', '_links', 'externals', 'genres', 'id', 'image', 'language', 'name', 'network', 'officialSite', 'premiered', 'rating', 'runtime', 'schedule', 'status', 'summary', 'type', 'updated', 'url', 'webChannel', 'weight']


In [7]:
shows.select(F.col('name')).show()
three_shows.select(F.col('type')).show()

+--------------+
|          name|
+--------------+
|Silicon Valley|
+--------------+

+--------+
|    type|
+--------+
|Scripted|
+--------+



In [8]:
# here "genres" is a pyspark complex type (meaning) it is a type containing another datastructure (in this case a list/array)
array_subset = three_shows.select(F.col('name'), F.col('genres'))
array_subset.show()

+--------------+--------+
|          name|  genres|
+--------------+--------+
|Silicon Valley|[Comedy]|
+--------------+--------+



In [9]:
array_subset.schema

StructType([StructField('name', StringType(), True), StructField('genres', ArrayType(StringType(), True), True)])

In [10]:
# you can access elements in a nested list via several routes
array_subset = array_subset.select(
    "name",
    array_subset.genres[0].alias("dot_and_index"),
    F.col("genres")[0].alias("col_and_index"),
    array_subset.genres.getItem(0).alias("dot_and_method"),
    F.col('genres').getItem(0).alias("col_and_method"),
)
array_subset.show()

+--------------+-------------+-------------+--------------+--------------+
|          name|dot_and_index|col_and_index|dot_and_method|col_and_method|
+--------------+-------------+-------------+--------------+--------------+
|Silicon Valley|       Comedy|       Comedy|        Comedy|        Comedy|
+--------------+-------------+-------------+--------------+--------------+



In [11]:
# performing multiple operations on an array column
array_subset_repeated = array_subset.select(
    "name",
    F.lit("Comedy").alias("one"),
    F.lit("Horror").alias("two"),
    F.lit("Drama").alias("three"),
    F.col("dot_and_index"),
)

array_subset_repeated.show(3, False)

array_subset_repeated = array_subset_repeated.select(
    "name",
    F.array("one", "two", "three").alias("Some_Genres"),
    F.array_repeat("dot_and_index", 5).alias("Repeated_Genres")
)

array_subset_repeated.show(3, False)

+--------------+------+------+-----+-------------+
|name          |one   |two   |three|dot_and_index|
+--------------+------+------+-----+-------------+
|Silicon Valley|Comedy|Horror|Drama|Comedy       |
+--------------+------+------+-----+-------------+

+--------------+-----------------------+----------------------------------------+
|name          |Some_Genres            |Repeated_Genres                         |
+--------------+-----------------------+----------------------------------------+
|Silicon Valley|[Comedy, Horror, Drama]|[Comedy, Comedy, Comedy, Comedy, Comedy]|
+--------------+-----------------------+----------------------------------------+



In [12]:
array_subset_repeated.select(
    "name",
    F.size("Some_Genres"),
    F.size("Repeated_Genres"),
).show()

+--------------+-----------------+---------------------+
|          name|size(Some_Genres)|size(Repeated_Genres)|
+--------------+-----------------+---------------------+
|Silicon Valley|                3|                    5|
+--------------+-----------------+---------------------+



In [13]:
array_subset_repeated.select(
    "name",
    F.array_distinct("Some_Genres"),
    F.array_distinct("Repeated_Genres"),
).show(truncate = False)

+--------------+---------------------------+-------------------------------+
|name          |array_distinct(Some_Genres)|array_distinct(Repeated_Genres)|
+--------------+---------------------------+-------------------------------+
|Silicon Valley|[Comedy, Horror, Drama]    |[Comedy]                       |
+--------------+---------------------------+-------------------------------+



In [14]:
# get common values by intersecting two arrays
array_subset_repeated = array_subset_repeated.select(
    "name",
    F.array_intersect("Some_Genres", "Repeated_Genres").alias("Genres")
)

array_subset_repeated.show(truncate=False)

+--------------+--------+
|name          |Genres  |
+--------------+--------+
|Silicon Valley|[Comedy]|
+--------------+--------+



In [15]:
# you can get the index of an array via
array_subset_repeated.select(
    "Genres",
    F.array_position("Genres", "Comedy")
).show()

+--------+------------------------------+
|  Genres|array_position(Genres, Comedy)|
+--------+------------------------------+
|[Comedy]|                             1|
+--------+------------------------------+



## 6.2.2 The map type: keys and values within a column

In [16]:
# construct a dataframe with a column containing a map
columns = ["name", "language", "type"]

shows_map = shows.select(
    *[F.lit(column) for column in columns],
    F.array(*columns).alias("values")
)

shows_map.show(truncate=False)

print(*[1,2,3,4])
print([1,2,3,4])

shows_map = shows_map.select(
    F.array(*columns).alias("keys"),
    "values"
)
shows_map.show(1, truncate=False)

shows_map = shows_map.select(
    F.map_from_arrays("keys", "values").alias("mapped")
)
shows_map.printSchema()
shows_map.show(1, False)



+----+--------+----+-----------------------------------+
|name|language|type|values                             |
+----+--------+----+-----------------------------------+
|name|language|type|[Silicon Valley, English, Scripted]|
+----+--------+----+-----------------------------------+

1 2 3 4
[1, 2, 3, 4]
+----------------------+-----------------------------------+
|keys                  |values                             |
+----------------------+-----------------------------------+
|[name, language, type]|[Silicon Valley, English, Scripted]|
+----------------------+-----------------------------------+

root
 |-- mapped: map (nullable = false)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)

+---------------------------------------------------------------+
|mapped                                                         |
+---------------------------------------------------------------+
|{name -> Silicon Valley, language -> English, type -> Scripted}|
+-------

In [17]:
# map property selection

shows_map.select(
    F.col("mapped.name"),
    F.col("mapped")["name"],
    shows_map.mapped["name"]
).show()

+--------------+--------------+--------------+
|          name|  mapped[name]|  mapped[name]|
+--------------+--------------+--------------+
|Silicon Valley|Silicon Valley|Silicon Valley|
+--------------+--------------+--------------+



In [18]:
import json 
import pprint 

# spark.read.json can't directly read json strings, it can only read from "files"
# e.g. the following throws an IllegalArgumentException
# sampleJson = spark.read.json("""
#     {"name": "Sample name","keywords": ["PySpark", "Python", "Data"]}
# """)

# to get around this we are going to wrap the JSON string in an RDD (resilient distributed dataset) which 
# can be read by spark.read.json

# create json as dictionary
ex06_1_json = {"name": "Sample name","keywords": ["PySpark", "Python", "Data"]}
# convert that dictionary into a json string
ex06_1_json = json.dumps(ex06_1_json)

#pretty print the json string
pprint.pprint(ex06_1_json)

# wrap the json string in a list and then pass it into `spark.sparkContext.parallelize` to wrap it in a rdd
sol6_1 = spark.read.json(spark.sparkContext.parallelize([ex06_1_json]))
sol6_1.printSchema()

'{"name": "Sample name", "keywords": ["PySpark", "Python", "Data"]}'
root
 |-- keywords: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- name: string (nullable = true)



In [19]:
import json
import pprint 

ex06_2_json = {"name": "Sample name", "keywords": ["Pyspark", 3.2, "Data"]}
ex06_2_json = json.dumps(ex06_2_json)

pprint.pprint(ex06_2_json)

sol6_2 = spark.read.json(spark.sparkContext.parallelize([ex06_2_json]))

sol6_2.printSchema() # this returns the same as ex06_1 as multi type arrays default to the "lowest common denominator" type. which in this case is a string
sol6_2.show()

'{"name": "Sample name", "keywords": ["Pyspark", 3.2, "Data"]}'
root
 |-- keywords: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- name: string (nullable = true)

+--------------------+-----------+
|            keywords|       name|
+--------------------+-----------+
|[Pyspark, 3.2, Data]|Sample name|
+--------------------+-----------+



## 6.3 The struct: Nesting columns within columns


In [20]:
shows.select("schedule").printSchema()
shows.select("schedule").show()

root
 |-- schedule: struct (nullable = true)
 |    |-- days: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- time: string (nullable = true)

+-----------------+
|         schedule|
+-----------------+
|{[Sunday], 22:00}|
+-----------------+



In [21]:
shows.select("_embedded").printSchema()
shows.head()

root
 |-- _embedded: struct (nullable = true)
 |    |-- episodes: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- _links: struct (nullable = true)
 |    |    |    |    |-- self: struct (nullable = true)
 |    |    |    |    |    |-- href: string (nullable = true)
 |    |    |    |-- airdate: string (nullable = true)
 |    |    |    |-- airstamp: string (nullable = true)
 |    |    |    |-- airtime: string (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- image: struct (nullable = true)
 |    |    |    |    |-- medium: string (nullable = true)
 |    |    |    |    |-- original: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- number: long (nullable = true)
 |    |    |    |-- runtime: long (nullable = true)
 |    |    |    |-- season: long (nullable = true)
 |    |    |    |-- summary: string (nullable = true)
 |    |    |    |-- url: string (nullable = true

Row(_embedded=Row(episodes=[Row(_links=Row(self=Row(href='http://api.tvmaze.com/episodes/10897')), airdate='2014-04-06', airstamp='2014-04-07T02:00:00+00:00', airtime='22:00', id=10897, image=Row(medium='http://static.tvmaze.com/uploads/images/medium_landscape/49/123633.jpg', original='http://static.tvmaze.com/uploads/images/original_untouched/49/123633.jpg'), name='Minimum Viable Product', number=1, runtime=30, season=1, summary="<p>Attending an elaborate launch party, Richard and his computer programmer friends - Big Head, Dinesh and Gilfoyle - dream of making it big. Instead, they're living in the communal Hacker Hostel owned by former programmer Erlich, who gets to claim ten percent of anything they invent there. When it becomes clear that Richard has developed a powerful compression algorithm for his website, Pied Piper, he finds himself courted by Gavin Belson, his egomaniacal corporate boss, who offers a $10 million buyout by his firm, Hooli. But Richard holds back when well-kno

In [22]:
# promote the embedded episodes to  top level
shows_clean = shows.withColumn("episodes", F.col("_embedded.episodes")).drop("_embedded")
shows_clean.printSchema()
shows_clean.show()

root
 |-- _links: struct (nullable = true)
 |    |-- previousepisode: struct (nullable = true)
 |    |    |-- href: string (nullable = true)
 |    |-- self: struct (nullable = true)
 |    |    |-- href: string (nullable = true)
 |-- externals: struct (nullable = true)
 |    |-- imdb: string (nullable = true)
 |    |-- thetvdb: long (nullable = true)
 |    |-- tvrage: long (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- id: long (nullable = true)
 |-- image: struct (nullable = true)
 |    |-- medium: string (nullable = true)
 |    |-- original: string (nullable = true)
 |-- language: string (nullable = true)
 |-- name: string (nullable = true)
 |-- network: struct (nullable = true)
 |    |-- country: struct (nullable = true)
 |    |    |-- code: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- timezone: string (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- name: string (nul

In [23]:
episodes_name = shows_clean.select(F.col("episodes.name"))
episodes_name.printSchema()
episodes_name.show()
episodes_name.select(F.explode("name").alias("name")).show(3, False)

root
 |-- name: array (nullable = true)
 |    |-- element: string (containsNull = true)

+--------------------+
|                name|
+--------------------+
|[Minimum Viable P...|
+--------------------+

+-------------------------+
|name                     |
+-------------------------+
|Minimum Viable Product   |
|The Cap Table            |
|Articles of Incorporation|
+-------------------------+
only showing top 3 rows



## 6.4 Building and using the data frame schema

 - Allows for faster data ingestion as spark won't have to read the data twice; once to generate schema and once to actually read the data.
 - Allows for fast failure in the case of data schema mismatch issues.

In [24]:
# a big enabler of this is the types package which allows us to specify schemas and types
import pyspark.sql.types as T 

# most types follow the "ValueType()" camel case convention from Java
T.LongType()
T.DecimalType(precision= 10, scale=0)
# complex types
T.ArrayType(T.StringType()) # specifies a list of strings
T.MapType(T.StringType(), T.LongType()) # specifies a map of strings mapping to longs
# Structs take a list of Struct fields
T.StructType([T.StructField(name= "fieldName", dataType=T.StringType(), nullable= False)])

# if you provide a reduced schema set when reading a wide data set you can save time as spark will skip the non-included fields.



StructType([StructField('fieldName', StringType(), False)])

In [25]:
# Simple example

coderSchema = T.StructType([
    T.StructField(name="devName", dataType= T.StringType()),
    T.StructField(name= "langs", dataType= T.ArrayType(T.StringType()))
])

coderData = [("Jane", ["Java", "Scala"]), ("Bob", ["Python", "Scala"])]

cdf = spark.createDataFrame(data = coderData, schema= coderSchema)

cdf.show()
cdf.printSchema()

cdf.select(F.col("devName")).show()
cdf.select(F.col("langs")[1]).show()

+-------+---------------+
|devName|          langs|
+-------+---------------+
|   Jane|  [Java, Scala]|
|    Bob|[Python, Scala]|
+-------+---------------+

root
 |-- devName: string (nullable = true)
 |-- langs: array (nullable = true)
 |    |-- element: string (containsNull = true)

+-------+
|devName|
+-------+
|   Jane|
|    Bob|
+-------+

+--------+
|langs[1]|
+--------+
|   Scala|
|   Scala|
+--------+



In [26]:
episode_links_schema = T.StructType([
    T.StructField(
        "self", T.StructType([
            T.StructField("href", T.StringType())
        ])
    )
])

episode_image_schema = T.StructType(
    [
        T.StructField("medium", T.StringType()),
        T.StructField("original", T.StringType()),
    ]
)  

episode_schema = T.StructType(
    [
        T.StructField("_links", episode_links_schema), # example of nested schema
        T.StructField("airdate", T.DateType()),
        T.StructField("airstamp", T.TimestampType()),
        T.StructField("airtime", T.StringType()),
        T.StructField("id", T.StringType()),
        T.StructField("image", episode_image_schema), # example of nested schema
        T.StructField("number", T.LongType()),
        T.StructField("runtime", T.LongType()),
        T.StructField("season", T.LongType()),
        T.StructField("summary", T.StringType()),
        T.StructField("url", T.StringType()),
    ]
)


embedded_schema = T.StructType(
    [
        T.StructField(
            "_embedded",
            T.StructType(
                [
                    T.StructField(
                        "episodes", T.ArrayType(episode_schema)
                    )
                ]
            ),
        )
    ]
)

In [27]:
#read with schema 
shows_with_schema = spark.read.json(
    "./data/shows/shows-silicon-valley.json",
    schema=embedded_schema,
    mode="FAILFAST" # tells the reader to crash if the input data is not compatible with our schema
)

shows_with_schema.show(n = 2, truncate=False, vertical=True)

-RECORD 0-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [28]:
# validate that the date based columns were successfully pulled in
for col in ["airdate", "airstamp"]:
    shows.select(f"_embedded.episodes.{col}").select(F.explode(col)).show(5, truncate=False)

+----------+
|col       |
+----------+
|2014-04-06|
|2014-04-13|
|2014-04-20|
|2014-04-27|
|2014-05-04|
+----------+
only showing top 5 rows

+-------------------------+
|col                      |
+-------------------------+
|2014-04-07T02:00:00+00:00|
|2014-04-14T02:00:00+00:00|
|2014-04-21T02:00:00+00:00|
|2014-04-28T02:00:00+00:00|
|2014-05-05T02:00:00+00:00|
+-------------------------+
only showing top 5 rows



In [29]:
# example of ingestion failure with with incompatible schema

from py4j.protocol import Py4JJavaError
import pyspark.sql.types as T 

bad_episode_schema = T.StructType([
    T.StructField("_links", episode_links_schema),
    T.StructField("airdate", T.DateType()),
    T.StructField("airstamp", T.TimestampType()),
    T.StructField("airtime", T.StringType()),
    T.StructField("id", T.StringType()),
    T.StructField("image", episode_image_schema),
    T.StructField("name", T.StringType()),
    T.StructField("number", T.LongType()),
    T.StructField("runtime", T.LongType()),
    T.StructField("season", T.LongType()),
    T.StructField("summary", T.LongType()),
    T.StructField("url", T.LongType()), # will cause an error on ingestion as a url is a string that is not compatible with a long
])

embedded_schema2 = T.StructType([
    T.StructField("_embedded",
                  T.StructType([
                      T.StructField("episodes", T.ArrayType(bad_episode_schema))
                  ])
                  )
])

shows_with_wrong_schema = spark.read.json(
    "./data/shows/shows-silicon-valley.json",
    schema=embedded_schema2,
    mode="FAILFAST",
)

try:
    shows_with_wrong_schema.show()
except Py4JJavaError as err:
    print(f"Unexpected error {err=}, {type(err)=}")


Unexpected error err=Py4JJavaError('An error occurred while calling o290.showString.\n', JavaObject id=o291), type(err)=<class 'py4j.protocol.Py4JJavaError'>


25/01/05 04:28:55 ERROR Executor: Exception in task 0.0 in stage 42.0 (TID 87)
org.apache.spark.SparkException: Encountered error while reading file file:///Users/hubert/Documents/dev/python/data-analysis-pyspark/data/shows/shows-silicon-valley.json. Details:
	at org.apache.spark.sql.errors.QueryExecutionErrors$.cannotReadFilesError(QueryExecutionErrors.scala:864)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.nextIterator(FileScanRDD.scala:296)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:131)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegenEvaluatorFac

In [30]:
from pprint import pprint
pprint(shows_with_schema.schema)
pprint(shows_with_schema.schema.json()) # this is a json string representation of the schema
pprint(shows_with_schema.schema.jsonValue()) # this is a python dictionary representation of the schema




StructType([StructField('_embedded', StructType([StructField('episodes', ArrayType(StructType([StructField('_links', StructType([StructField('self', StructType([StructField('href', StringType(), True)]), True)]), True), StructField('airdate', DateType(), True), StructField('airstamp', TimestampType(), True), StructField('airtime', StringType(), True), StructField('id', StringType(), True), StructField('image', StructType([StructField('medium', StringType(), True), StructField('original', StringType(), True)]), True), StructField('number', LongType(), True), StructField('runtime', LongType(), True), StructField('season', LongType(), True), StructField('summary', StringType(), True), StructField('url', StringType(), True)]), True), True)]), True)])
'{"fields":[{"metadata":{},"name":"_embedded","nullable":true,"type":{"fields":[{"metadata":{},"name":"episodes","nullable":true,"type":{"containsNull":true,"elementType":{"fields":[{"metadata":{},"name":"_links","nullable":true,"type":{"field

In [31]:
from pprint import pprint
pprint(shows_with_schema.select(F.explode("_embedded.episodes").alias("episode"))
       .select("episode.airtime")
       .schema.jsonValue()
       )

{'fields': [{'metadata': {},
             'name': 'airtime',
             'nullable': True,
             'type': 'string'}],
 'type': 'struct'}


In [32]:
pprint(T.StructField("array_example", T.ArrayType(T.StringType())).jsonValue())
pprint(T.StructField("map_example", T.MapType(T.StringType(), T.LongType())).jsonValue())
pprint(T.StructType([T.StructField(name="array_field", dataType=T.ArrayType(T.StringType())),
                     T.StructField(name="map_field", dataType=T.MapType(T.StringType(), T.LongType()))
                     ]).jsonValue())


{'metadata': {},
 'name': 'array_example',
 'nullable': True,
 'type': {'containsNull': True, 'elementType': 'string', 'type': 'array'}}
{'metadata': {},
 'name': 'map_example',
 'nullable': True,
 'type': {'keyType': 'string',
          'type': 'map',
          'valueContainsNull': True,
          'valueType': 'long'}}
{'fields': [{'metadata': {},
             'name': 'array_field',
             'nullable': True,
             'type': {'containsNull': True,
                      'elementType': 'string',
                      'type': 'array'}},
            {'metadata': {},
             'name': 'map_field',
             'nullable': True,
             'type': {'keyType': 'string',
                      'type': 'map',
                      'valueContainsNull': True,
                      'valueType': 'long'}}],
 'type': 'struct'}


In [33]:
# validating that schema matches another schema

schema1 = T.StructType.fromJson(json.loads(shows_with_schema.schema.json()))
print(shows_with_schema.schema == schema1)

True


## 6.5 Reducing Duplicate Data with complex data types
AKA database normalization but for spark
Each normalized record with a unique primary key is an "exposure record"

Complex data types allow for the simplicity of a single table but without the costs (data duplication and unclear relationships) of denormalization
Kind of like a list of Java Objects...

E.g. If each record row in a table represents a show.
 - The episodes of a show can be represented as an array of structs in a single column of the show record row.
 - The properties of each episode can be represented as fields in the episode struct
 - Each show can have multiple genres represented by a single array of strings.
 - Each show can have a schedule represented by struct column
 - Each schedule can have multiple days (array) but a single time column



### 6.5.1 Explode and Collect to temporarily denormalize and renormalize dataframes


In [34]:
episodes = shows.select(F.col("id"), F.explode("_embedded.episodes").alias("episode"))
episodes.show()

+---+--------------------+
| id|             episode|
+---+--------------------+
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
|143|{{{http://api.tvm...|
+---+--------------------+
only showing top 20 rows



In [35]:
episode_name_id_map = shows.select(
    F.map_from_arrays(
        F.col("_embedded.episodes.id"),
        F.col("_embedded.episodes.name")
    ).alias("name_id")
)
episode_name_id_map.show(truncate=False)

episode_name_id_map.select(F.posexplode("name_id").alias("pos", "id", "name")).show()

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [36]:
collected = episodes.groupBy("id").agg(F.collect_list("episode").alias("episodes"))
collected.show(1, truncate=False, vertical=True)

-RECORD 0-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [37]:
epNames = shows.select(F.col("id"), F.explode("_embedded.episodes.name").alias("name"))
epNames.show()

cNames = epNames.groupBy("id").agg(F.collect_list("name").alias("names"))
cNames.show(truncate=False)

+---+--------------------+
| id|                name|
+---+--------------------+
|143|Minimum Viable Pr...|
|143|       The Cap Table|
|143|Articles of Incor...|
|143|    Fiduciary Duties|
|143|      Signaling Risk|
|143|Third Party Insou...|
|143|    Proof of Concept|
|143|Optimal Tip-to-Ti...|
|143|   Sand Hill Shuffle|
|143| Runaway Devaluation|
|143|           Bad Money|
|143|            The Lady|
|143|        Server Space|
|143|            Homicide|
|143|       Adult Content|
|143| White Hat/Black Hat|
|143| Binding Arbitration|
|143|Two Days of the C...|
|143|    Founder Friendly|
|143|      Two in the Box|
+---+--------------------+
only showing top 20 rows

+---+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [38]:
pprint(shows.select(F.col("_embedded.episodes")).schema)

name_runtime = shows.select(F.col("id"))

StructType([StructField('episodes', ArrayType(StructType([StructField('_links', StructType([StructField('self', StructType([StructField('href', StringType(), True)]), True)]), True), StructField('airdate', StringType(), True), StructField('airstamp', StringType(), True), StructField('airtime', StringType(), True), StructField('id', LongType(), True), StructField('image', StructType([StructField('medium', StringType(), True), StructField('original', StringType(), True)]), True), StructField('name', StringType(), True), StructField('number', LongType(), True), StructField('runtime', LongType(), True), StructField('season', LongType(), True), StructField('summary', StringType(), True), StructField('url', StringType(), True)]), True), True)])


In [39]:
#Creating your own structs
struct_ex = shows.select(
    F.struct(
        F.col("status"),
        F.col("weight"),
        F.lit(True).alias("has_watched")
    ).alias("info")
)

In [40]:
struct_ex.show()
struct_ex.printSchema()

+-----------------+
|             info|
+-----------------+
|{Ended, 96, true}|
+-----------------+

root
 |-- info: struct (nullable = false)
 |    |-- status: string (nullable = true)
 |    |-- weight: long (nullable = true)
 |    |-- has_watched: boolean (nullable = false)



In [42]:
dict_schema = T.StructType([
    T.StructField("one", T.IntegerType()),
    T.StructField("two", T.ArrayType(T.IntegerType()))
])
a = spark.createDataFrame([{"one": 1, "two": [1,2,3]}], schema=dict_schema)
a.show()

+---+---------+
|one|      two|
+---+---------+
|  1|[1, 2, 3]|
+---+---------+



In [44]:
three_shows.show()
three_shows.printSchema()


+--------------------+--------------------+--------------------+--------+---+--------------------+--------+--------------+--------------------+--------------------+----------+------+-------+-----------------+------+--------------------+--------+----------+--------------------+----------+------+
|           _embedded|              _links|           externals|  genres| id|               image|language|          name|             network|        officialSite| premiered|rating|runtime|         schedule|status|             summary|    type|   updated|                 url|webChannel|weight|
+--------------------+--------------------+--------------------+--------+---+--------------------+--------+--------------+--------------------+--------------------+----------+------+-------+-----------------+------+--------------------+--------+----------+--------------------+----------+------+
|{[{{{http://api.t...|{{http://api.tvma...|{tt2575988, 27716...|[Comedy]|143|{http://static.tv...| English|Silic

In [ ]:
## 6.6 get the tenure in days for each show
three_shows.select(
    F.col("name"), 
    F.array_min(F.col("_embedded.episodes.airdate")).cast("date").alias("min_airdate"),
    F.array_max(F.col("_embedded.episodes.airdate")).cast("date").alias("max_airdate")
    ).select(
    F.col("name"),
    F.datediff(F.col("max_airdate"), F.col("min_airdate")).alias("days_between"),
    (F.col("max_airdate") - F.col("min_airdate")).alias("tenure")
    ).show()


+--------------+------------+-------------------+
|          name|days_between|             tenure|
+--------------+------------+-------------------+
|Silicon Valley|        2072|INTERVAL '2072' DAY|
+--------------+------------+-------------------+



In [57]:
## Q6.7 Take the shows data frame and extract the air date and name of each episode in two array columns.
shows.show()
# shows.printSchema()
shows.select(
    F.col("name"),
    F.col("_embedded.episodes.airdate"),
    F.col("_embedded.episodes.name")
).show()

+--------------------+--------------------+--------------------+--------+---+--------------------+--------+--------------+--------------------+--------------------+----------+------+-------+-----------------+------+--------------------+--------+----------+--------------------+----------+------+
|           _embedded|              _links|           externals|  genres| id|               image|language|          name|             network|        officialSite| premiered|rating|runtime|         schedule|status|             summary|    type|   updated|                 url|webChannel|weight|
+--------------------+--------------------+--------------------+--------+---+--------------------+--------+--------------+--------------------+--------------------+----------+------+-------+-----------------+------+--------------------+--------+----------+--------------------+----------+------+
|{[{{{http://api.t...|{{http://api.tvma...|{tt2575988, 27716...|[Comedy]|143|{http://static.tv...| English|Silic

In [62]:
shows.select(
    F.explode(F.col("_embedded.episodes")).alias("episodes")
).select(
    F.col("episodes.airdate"),
    F.col("episodes.name")
).show(truncate=False)

+----------+-----------------------------+
|airdate   |name                         |
+----------+-----------------------------+
|2014-04-06|Minimum Viable Product       |
|2014-04-13|The Cap Table                |
|2014-04-20|Articles of Incorporation    |
|2014-04-27|Fiduciary Duties             |
|2014-05-04|Signaling Risk               |
|2014-05-11|Third Party Insourcing       |
|2014-05-18|Proof of Concept             |
|2014-06-01|Optimal Tip-to-Tip Efficiency|
|2015-04-12|Sand Hill Shuffle            |
|2015-04-19|Runaway Devaluation          |
|2015-04-26|Bad Money                    |
|2015-05-03|The Lady                     |
|2015-05-10|Server Space                 |
|2015-05-17|Homicide                     |
|2015-05-24|Adult Content                |
|2015-05-31|White Hat/Black Hat          |
|2015-06-07|Binding Arbitration          |
|2015-06-14|Two Days of the Condor       |
|2016-04-24|Founder Friendly             |
|2016-05-01|Two in the Box               |
+----------

In [73]:
## Q6.8 Given the following data frame, create a new data frame that contains a single map from one to square 

exo6_8 = spark.createDataFrame([[1, 2], [2, 4], [3, 9]], ["one", "square"])
exo6_8.show()

exo6_8.select(
    F.map_from_arrays(
        F.collect_list("one"),
        F.collect_list("square")
    ).alias("mapOneToSquare")
).show(truncate=False)

exo6_8.select(
    F.map_from_entries(
        F.collect_list(F.struct("one", "square")),
    ).alias("mapOneToSquare")
).show(truncate=False)

exo6_8.groupBy().agg(
    F.map_from_entries(
        F.collect_list(
            F.struct(F.col("one"), F.col("square"))
        )
    ).alias("mapOneToSquare")
).show(truncate=False)

exo6_8.groupBy().agg(
    F.collect_list("one").alias("ones"),
    F.collect_list("square").alias("squares")
).select(
    F.map_from_arrays(F.col("ones"), F.col("squares")).alias("mapOneToSquare")
).show(truncate=False)

+---+------+
|one|square|
+---+------+
|  1|     2|
|  2|     4|
|  3|     9|
+---+------+

+------------------------+
|mapOneToSquare          |
+------------------------+
|{1 -> 2, 2 -> 4, 3 -> 9}|
+------------------------+

+------------------------+
|mapOneToSquare          |
+------------------------+
|{1 -> 2, 2 -> 4, 3 -> 9}|
+------------------------+

+------------------------+
|mapOneToSquare          |
+------------------------+
|{1 -> 2, 2 -> 4, 3 -> 9}|
+------------------------+

+------------------------+
|mapOneToSquare          |
+------------------------+
|{1 -> 2, 2 -> 4, 3 -> 9}|
+------------------------+

